<img src="../img/GTK_Logo_Social Icon.jpg" width=175 align="right" />

# Worksheet 4.2: Tuning your Classifier - Answers
This worksheet covers concepts relating to tuning a classifier.  It should take no more than 20-30 minutes to complete.  Please raise your hand if you get stuck.  

## Import the Libraries
For this exercise, we will be using:
* Pandas (https://pandas.pydata.org/pandas-docs/stable/)
* Numpy (https://docs.scipy.org/doc/numpy/reference/)
* Scikit-learn (https://scikit-learn.org/stable/documentation.html)

In [1]:
# Load Libraries - Make sure to run this cell!
import pandas as pd
import numpy as np
import time
import pickle
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

import warnings; warnings.simplefilter('ignore')
DATA_HOME = '../data'

## Load the Data
For this exercise, we are going to focus on building a pipeline and then tuning the resultant model.

In [2]:
df_final = pd.read_csv(f'{DATA_HOME}/dga_features_final_df.csv')
target = df_final['isDGA']
feature_matrix = df_final.drop(['isDGA'], axis='columns')
feature_matrix.sample(5)

,length,digits,entropy,vowel-cons,firstDigitIndex,ngrams
1842,2,1,1.000000,0.000000,2,336.833333
400,12,0,3.418296,0.333333,0,944.233838
628,27,0,3.736007,0.173913,0,746.578727
1803,13,0,3.085055,0.444444,0,1657.296426
449,9,0,3.169925,0.800000,0,1591.238757


### Split the data into training and testing sets.
We're going to need a training and testing dataset, so you know the drill, split the data..

In [3]:
feature_matrix_train, feature_matrix_test, target_train, target_test = train_test_split(feature_matrix, 
                                                                                        target, 
                                                                                        test_size=0.25)

## Build a Model
For this exercise, we're going to create a K-NN Classifier for the DGA data and tune it.   (http://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html) 


1. Create a classifier with the default options (leave the arguments blank and they will use the defaults).
2. Train the classifier on the training data
3. Calculate the accuracy score for the model based on the test data.


The sklearn default values for the KNeighborsClassifier hyperparameters are shown below.
```python 
KNeighborsClassifier(algorithm='auto', 
                     leaf_size=30, 
                     metric='minkowski',
                     metric_params=None, 
                     n_jobs=1, 
                     n_neighbors=5, 
                     p=2,
                     weights='uniform')
```           

In [4]:
#create
knn_model = KNeighborsClassifier()
#train
knn_model.fit( feature_matrix_train, target_train )

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[object](2,)","['dga','legit']"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'


In [5]:
#predict
default_predictions = knn_model.predict(feature_matrix_test)

In [6]:
#metric
accuracy_score(target_test, default_predictions)

0.826

In [7]:
# save the model
filename = f'{DATA_HOME}/dga_model.sav'
pickle.dump(knn_model, open(filename, 'wb'))

## Improving Performance 
The model achieves approximately 85% accuracy when we use the default parameters.  It is significantly better than chance (50%) but let's see if we can do better. 

**Note:  This notebook is written without using fixed random seeds, so you might get slightly different results.**

### Scaling the Features (preprocessing)
K-NN is a distance-based classifier and hence it is necessary to scale the features prior to training the model.  

### Create Pipeline
Pipeline allows you to sequentially apply a list of transformers to preprocess the data and, if desired, conclude the sequence with a final predictor for predictive modeling.  Let's create a simple pipeline with two steps:

1.  StandardScaler
2.  Train the classifier

Once you've done that, calculate the accuracy and see if it has improved.

In [8]:
pipeline_knn = Pipeline([
    ('scaler',StandardScaler()),
    ('knn_model', KNeighborsClassifier())
])

pipeline_knn.fit(feature_matrix_train, target_train ).score(feature_matrix_test, target_test)

0.882

Scaling the features did result in a small improvement: .85 accuracy to .88.  But let's see if we can't do even better.

### Using RandomSearchCV and GridSearchCV to tune Hyperparameters
Now that we've scaled the features and built a simple pipeline, let's try to tune the hyperparameters to see if we can improve the model performance.  Scikit-learn provides two methods for accomplishing this task: `RandomizedSearchCV` and `GridSearchCV`. 


* `GridSearchCV`:  GridSearch iterates through all possible combinations of tuning parameters to find the optimal combination. (http://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)
* `RandomizedSearchCV`:  RandomizedSearch iterates through random combinations of parameters to find the optimal combination.  While RandomizedSearch does not try every possible combination, is considerably faster than GridSearch and has been shown to get very close to the optimal combination in considerably less time.  (http://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html)

Both `RandomizedSearchCV` and `GridSearchCV` require you to provide a grid of parameters.  You will need to refer to the documentation for the classifier you are using to get a list of parameters for that particular model.  Also since we will be using the pipeline, you have to format the parameters correctly.  The name of the variable must be preceeded by the name of the step in your pipeline and two underscores.  For example.  If the classifier in the pipeline is called `knn_model`, and you have a tuning parameter called `metric`, the parameter grid would be as follows:
```python
params = {
    "knn_model__n_neighbors": np.arange(1, 50, 2),
    "knn_model__metric": ["euclidean", "cityblock"] 
}
```
### Your Task
Using RandomizedSearchCV, improve the performance of your model.

In [9]:
params = {"knn_model__n_neighbors": np.arange(1, 50, 2), 
         "knn_model__weights": ["uniform", "distance"],
         "knn_model__algorithm": ['auto', 'ball_tree', 'kd_tree', 'brute'],
         "knn_model__leaf_size": np.arange(1, 80, 2),
         "knn_model__p": [1,2],
         "knn_model__metric": ["euclidean", "manhattan"]}


grid = RandomizedSearchCV(pipeline_knn, params, n_iter=100)
start = time.time()
grid.fit(feature_matrix_train, target_train)
 
# evaluate the best randomized searched model on the test data
print("[INFO] randomized search took {:.2f} seconds".format(time.time() - start))

acc = grid.best_score_
print("[INFO] grid search accuracy: {:.2f}%".format(acc * 100))
print("[INFO] randomized search best parameters: {}".format(grid.best_params_))

[INFO] randomized search took 2.69 seconds
[INFO] grid search accuracy: 90.47%
[INFO] randomized search best parameters: {'knn_model__weights': 'uniform', 'knn_model__p': 1, 'knn_model__n_neighbors': np.int64(17), 'knn_model__metric': 'manhattan', 'knn_model__leaf_size': np.int64(13), 'knn_model__algorithm': 'ball_tree'}


the model was able to achieve an improved accuracy with RandomSearch!   


## Model Comparison
Your final task is to:
1.  Using RandomForest, create a classifier for the DGA dataset
2.  Use either GridSearchCV or RandomizedSearchCV to find the optimal parameters for this model.

How does this model compare with the first K-NN classifier for this data?

In [10]:
rf_clf = RandomForestClassifier()
params = {
    "n_estimators": np.arange(1, 400, 50),
    "max_features": ['auto', 'sqrt','log2' ],
    "max_depth": np.arange(1, 20, 2),
    "criterion": ['gini','entropy']
} 

rf_grid = RandomizedSearchCV(rf_clf, params )
start = time.time()
rf_grid.fit(feature_matrix_train, target_train)
 
# evaluate the best randomized searched model on the testing
# data
print("[INFO] randomized search took {:.2f} seconds".format(time.time() - start))

#acc = grid.score(feature_matrix_test, target_test)
acc = rf_grid.best_score_
print("[INFO] grid search accuracy: {:.2f}%".format(acc * 100))
print("[INFO] randomized search best parameters: {}".format(rf_grid.best_params_))

[INFO] randomized search took 2.31 seconds
[INFO] grid search accuracy: 90.60%
[INFO] randomized search best parameters: {'n_estimators': np.int64(201), 'max_features': 'sqrt', 'max_depth': np.int64(3), 'criterion': 'gini'}
